# Session 10 — Duality & Sinkhorn

**Author:** PPSP lab  **·**  **Date:** 2026-05-27  **·**  **Project:** Quantum Optimal Transport course  **·**  **Status:** 🔬 Teaching

## Purpose

S10 is the **bridge session** of the course. We meet:

1. **Kantorovich duality** — the dual face of the OT LP, with **1-Lipschitz potentials** for $W_1$.
2. **Entropic regularization** — add an entropy term to make the LP **strongly convex**, smooth, and *fast*.
3. **The Sinkhorn algorithm** (Sinkhorn 1964; Cuturi 2013) — five lines of matrix scaling that revolutionised computational OT.
4. **Amari's information-geometric reading**: *the Sinkhorn plan is a **KL projection** of the Gibbs kernel onto the transportation polytope*. **Wasserstein meets KL.**

This is also the bridge that survives the lift to quantum OT (S14).

## 0. What you'll be able to do

- State the **Kantorovich dual LP** and verify primal–dual equality numerically.
- Recognise the **1-Lipschitz dual of $W_1$** (Kantorovich–Rubinstein).
- Implement **Sinkhorn in five lines** and run it on real data.
- Read the **ε trade-off** between sharp ($\to$ LP) and blurry ($\to$ independent coupling) plans.
- **Verify Amari's bridge** numerically: the Sinkhorn plan is the KL projection of the Gibbs kernel onto $T(a, b)$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ot

from qot_course import viz
from qot_course.transport.discrete import squared_euclidean_cost, transport_cost
from qot_course.transport.sinkhorn import (
    gibbs_kernel, sinkhorn, sinkhorn_distance,
    sinkhorn_iterations_log, kl_to_kernel,
)

np.random.seed(42)
viz.use_course_style()

## 1. Kantorovich duality — the dual face of the LP

Every linear program has a **dual**. For the Kantorovich primal

$$ \min_{P \ge 0,\ P\mathbf{1} = a,\ P^\top\mathbf{1} = b}\ \langle C, P\rangle, $$

the dual reads

$$
\boxed{\;\max_{\varphi \in \mathbb{R}^n,\ \psi \in \mathbb{R}^m}\
 \langle a, \varphi\rangle + \langle b, \psi\rangle
 \quad \text{subject to} \quad
 \varphi_i + \psi_j \le C_{ij} \;\;\forall i, j. \;}
$$

The dual variables $\varphi, \psi$ are called **Kantorovich potentials**. By LP duality
(no gap; both polytopes are bounded), the dual maximum equals the primal minimum. Let's
verify it numerically.

**Special case (Kantorovich–Rubinstein).** For the ground cost $c(x, y) = |x - y|$, the
dual constraint forces $\varphi(x) - \varphi(y) \le |x - y|$ (with $\psi = -\varphi$).
This is the **1-Lipschitz condition**, and the dual becomes

$$ W_1(\mu, \nu) = \sup_{\mathrm{Lip}(f) \le 1} \int f\,\mathrm{d}\mu - \int f\,\mathrm{d}\nu. $$

A 1-Lipschitz test function ranks distributions by how much mass they shift along the
ground space. We will see in S15 that this duality survives the lift to density matrices.

In [ ]:
# Small problem: 4 source atoms, 5 target atoms, squared cost.
rng = np.random.default_rng(0)
a = rng.random(4); a = a / a.sum()
b = rng.random(5); b = b / b.sum()
cost = squared_euclidean_cost(np.arange(4, dtype=float), np.linspace(0, 3, 5))

# Solve and ask POT for the dual potentials.
plan, log_info = ot.emd(a, b, cost, log=True)
phi, psi = log_info["u"], log_info["v"]

primal_value = float(np.sum(cost * plan))
dual_value   = float(a @ phi + b @ psi)
max_violation = float(np.max(phi[:, None] + psi[None, :] - cost))

print(f"primal  <C, P*>           = {primal_value:.6f}")
print(f"dual    <a, phi> + <b, psi> = {dual_value:.6f}")
print(f"gap                          = {abs(primal_value - dual_value):.2e}    (should be 0)")
print(f"max dual constraint violation = {max_violation:+.2e}  (should be <= 0)")

**Read the output.** Primal and dual values agree to machine precision (**no duality
gap**). The dual constraint $\varphi_i + \psi_j \le C_{ij}$ is satisfied for every pair
$(i, j)$. The LP is "doubly solved" — one number, two stories.

The downside: even with this beautiful structure, the LP is $\mathcal{O}(n^3)$. For
data-scale problems ($n \sim 10^5$ in images or single-cell genomics) that is
impractical. We need a faster solver. Enter **entropic regularization**.

## 2. Entropic regularization — making the LP strongly convex

Cuturi (2013) proposed adding a small entropy bonus to the OT cost:

$$
\boxed{\;P^\star_\varepsilon \,=\, \arg\min_{P \in T(a, b)}\
       \langle C, P\rangle - \varepsilon\, H(P),\qquad
       H(P) = -\sum_{i, j} P_{ij}\log P_{ij}. \;}
$$

Why? Three reasons:

- **Strong convexity** ($H$ is strictly concave) $\Rightarrow$ the minimizer is *unique* and changes smoothly with the data.
- **Multiplicative structure**: the KKT conditions force $P^\star_\varepsilon$ to have the form $u_i\, K_{ij}\, v_j$ with the **Gibbs kernel** $K_{ij} = e^{-C_{ij}/\varepsilon}$.
- **Cheap iterations**: the $u, v$ scaling factors can be found by **alternating matrix–vector products** — no LP solver needed.

Letting $\varepsilon \to 0$ recovers the LP optimum. Letting $\varepsilon \to \infty$
shrinks all of $K$ toward the constant matrix and yields the **independent coupling**
$P = a \otimes b$.

## 3. The Sinkhorn algorithm — five lines

The KKT conditions of the entropic problem give:

$$ P^\star_\varepsilon = \mathrm{diag}(u)\,K\,\mathrm{diag}(v), $$

with $u, v$ determined by the marginal constraints $\mathrm{diag}(u)(Kv) = a$ and
$\mathrm{diag}(v)(K^\top u) = b$. Sinkhorn's iteration just alternates them:

```python
K = exp(-C / epsilon)
u = ones(n); v = ones(m)
while not converged:
    v = b / (K.T @ u)
    u = a / (K @ v)
```

That is the **entire algorithm**. It is provably geometrically convergent (Franklin &
Lorenz 1989), and at each step is dominated by two matrix–vector products
($\mathcal{O}(nm)$ per iteration vs $\mathcal{O}(n^3)$ for the LP).

In [ ]:
# Solve the same small problem entropically, with a moderate epsilon.
epsilon = 0.5
plan_sk = sinkhorn(a, b, cost, epsilon=epsilon, n_iter=2000, tol=1e-12)

print(f"row sums  P 1:  {np.round(plan_sk.sum(axis=1), 6).tolist()}")
print(f"target    a:    {np.round(a, 6).tolist()}")
print(f"col sums  P^T 1:{np.round(plan_sk.sum(axis=0), 6).tolist()}")
print(f"target    b:    {np.round(b, 6).tolist()}")
print()

# Compare to the LP optimum (sharp) and to POT's sinkhorn.
lp_cost = float(ot.emd2(a, b, cost))
sk_cost = transport_cost(plan_sk, cost)
print(f"LP cost           <C, P*_0>        = {lp_cost:.4f}")
print(f"Sinkhorn cost     <C, P*_eps>      = {sk_cost:.4f}    (eps = {epsilon})")
print(f"agreement with POT (sanity check): {np.allclose(plan_sk, ot.sinkhorn(a, b, cost, epsilon, numItermax=2000, stopThr=1e-12), atol=1e-4)}")

**Read the output.** Marginals match $a$ and $b$ to floating-point precision: the
Sinkhorn plan is feasible. The entropic transport cost is **strictly larger** than the
LP optimum (because the entropy regularisation *biases* the plan toward more spread-out
couplings); the gap shrinks as $\varepsilon \to 0$. And our 5-line implementation agrees
with POT's reference solver.

In [ ]:
# Marginal-error diagnostic over Sinkhorn iterations.
err_row, _ = sinkhorn_iterations_log(a, b, cost, epsilon=epsilon, n_iter=200)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.semilogy(err_row, color=viz.SOURCE_COLOR, lw=2)
ax.set_xlabel("Sinkhorn iteration")
ax.set_ylabel(r"row-marginal max error  $\|P\,\mathbf{1} - a\|_\infty$  (log scale)")
ax.set_title(f"Geometric convergence of Sinkhorn  (eps = {epsilon})", pad=12)
ax.axhline(1e-9, color=viz.TARGET_COLOR, linestyle="--", alpha=0.7, label="machine-noise floor (~$10^{-9}$)")
ax.legend()
plt.show()

**Read the figure.** Straight line on a log scale = **geometric (exponential)
convergence**. The rate depends on $\varepsilon$ and the cost matrix; at this
$\varepsilon = 0.5$ we reach machine precision in $\sim 200$ iterations on a $4 \times 5$
problem. (For smaller $\varepsilon$ the rate worsens and the standard implementation may
underflow — use **log-domain Sinkhorn** in that regime.)

## 4. The ε trade-off — sharp vs blurry

The entropy term blurs the plan. Tiny $\varepsilon$ keeps it sparse (close to the LP);
large $\varepsilon$ pushes it toward the independent coupling. Watch the same problem
at four values of $\varepsilon$.

In [ ]:
# A nicer-looking 1-D example with smooth bumps to see the diagonal structure.
positions = np.linspace(0.0, 9.0, 16)
def bump(c, sigma=1.2):
    p = np.exp(-0.5 * ((positions - c) / sigma) ** 2)
    return p / p.sum()
src = bump(2.0)
tgt = bump(7.0)
C_demo = squared_euclidean_cost(positions, positions)

epsilons = [0.05, 0.5, 5.0, 50.0]
plans = [sinkhorn(src, tgt, C_demo, epsilon=eps, n_iter=2000, tol=1e-10)
         for eps in epsilons]

fig, axes = plt.subplots(1, 4, figsize=(13, 3.6))
for ax, plan, eps in zip(axes, plans, epsilons):
    im = ax.imshow(plan, cmap=viz.CMAP_PLAN, origin="lower", aspect="equal")
    ax.set_title(rf"$\varepsilon = {eps}$", pad=8)
    ax.set_xlabel("target position"); ax.grid(False)
    if ax is axes[0]:
        ax.set_ylabel("source position")
    fig.colorbar(im, ax=ax, shrink=0.85)
fig.suptitle("Entropic transport plans across $\\varepsilon$: sharp diagonal $\\to$ blurry independent coupling",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

**Read the four heatmaps.**

- $\varepsilon = 0.05$: a **sharp diagonal stripe** — essentially the LP solution; mass moves the minimum distance.
- $\varepsilon = 0.5$: still diagonal, but slightly **blurred** along the off-diagonal.
- $\varepsilon = 5.0$: the diagonal structure is mostly gone, replaced by a **smooth bilinear pattern** ($\approx u_i v_j$).
- $\varepsilon = 50.0$: nearly the **independent coupling** $\mathrm{src} \otimes \mathrm{tgt}$ — the entropy bonus has dominated the transport cost.

That smooth sweep is the **ε–smoothness ladder**: in practice we pick $\varepsilon$
small enough to keep the geometric content and large enough to keep the algorithm fast
and stable. It is a *regularisation trade-off* just like in any machine-learning task.

## 5. Amari's bridge — Sinkhorn is a KL projection

Here is the most beautiful identity of the entire course. Plug the entropic objective
into the KL divergence with the **Gibbs kernel** $K_{ij} = e^{-C_{ij}/\varepsilon}$:

$$
\varepsilon\,\mathrm{KL}(P \,\|\, K)
= \sum_{ij} P_{ij}\bigl(\varepsilon\log P_{ij} + C_{ij}\bigr) + \mathrm{const}
= \langle C, P\rangle - \varepsilon\, H(P) + \mathrm{const}.
$$

So **minimising the entropic OT cost over $T(a, b)$ is the same as minimising
$\mathrm{KL}(P \,\|\, K)$ over $T(a, b)$**. In one line: the Sinkhorn plan is the
**KL projection of the Gibbs kernel onto the transportation polytope**.

This is **Wasserstein meets KL** — the M3 transport geometry intersects the M2
information geometry, exactly as Amari (2016, sec. 7.5) describes. The Sinkhorn
iterations are *iterative Bregman projections* under the KL divergence: each step is a
multiplicative rescaling of rows or columns (a property of the KL Bregman divergence).

Let's verify the projection property numerically.

In [ ]:
# At a fixed eps, the Sinkhorn plan must minimise KL(. || K) over T(a, b).
# Any marginal-preserving perturbation should *increase* the KL.
eps = 0.4
K   = gibbs_kernel(C_demo, eps)
plan_star = sinkhorn(src, tgt, C_demo, epsilon=eps, n_iter=5000, tol=1e-14)
kl_star = kl_to_kernel(plan_star, K)
print(f"KL(P*_eps || K) at the Sinkhorn fixed point = {kl_star:.6f}")
print()

# Try 5 marginal-preserving 2x2-cycle perturbations and confirm KL increases.
rng = np.random.default_rng(7)
for trial in range(5):
    i1, i2 = rng.choice(plan_star.shape[0], 2, replace=False)
    j1, j2 = rng.choice(plan_star.shape[1], 2, replace=False)
    E = np.zeros_like(plan_star)
    E[i1, j1] = 1.0; E[i2, j2] = 1.0
    E[i1, j2] = -1.0; E[i2, j1] = -1.0
    delta = 0.5 * min(plan_star[i1, j2], plan_star[i2, j1])
    if delta < 1e-12:
        continue
    perturbed = plan_star + delta * E
    if np.all(perturbed > 0):
        kl_pert = kl_to_kernel(perturbed, K)
        gap = kl_pert - kl_star
        rows_ok = np.allclose(perturbed.sum(axis=1), src, atol=1e-12)
        cols_ok = np.allclose(perturbed.sum(axis=0), tgt, atol=1e-12)
        print(f"trial {trial+1}: marginals preserved? {rows_ok and cols_ok}    "
              f"KL increase = {gap:+.6e}   (must be > 0)")

**Read the output.** Every marginal-preserving perturbation we tried makes
$\mathrm{KL}(P \,\|\, K)$ **strictly larger** than at the Sinkhorn fixed point — i.e.
$P^\star_\varepsilon$ is a (local, and in fact global) **minimiser** of
$\mathrm{KL}(\cdot \,\|\, K)$ on the transportation polytope. **Amari's bridge holds
numerically.**

Why is this *more* than a cute reformulation?
- **It tells us how to solve entropic OT in any geometry that admits a KL divergence.**
  In S14 the same bridge will give us **quantum Sinkhorn** with the *Umegaki* relative
  entropy replacing the classical KL.
- **It clarifies why Sinkhorn is fast.** Iterative Bregman projections on affine
  constraints converge linearly under KL — that's the rate we just saw.
- **It clarifies why $\varepsilon$ matters.** Smaller $\varepsilon \Rightarrow$ sharper
  $K$, harder projection, slower convergence. Larger $\varepsilon \Rightarrow$ smoother
  $K$, easier projection, but biased away from the true OT plan.

The transport geometry and the information geometry of S6/S7 are the **same geometry**
seen from two angles. From here, quantum OT is just one more change of object.

## 6. Dictionary update — Sinkhorn closes the M2 ↔ M3 loop

| Classical | Quantum |
|-----------|---------|
| probability vector $a$ | density matrix $\rho$ |
| KL divergence $D(p\|q)$ | Umegaki $S(\rho\|\sigma)$ |
| Wasserstein-$p$ distance $W_p$ | quantum Wasserstein *(S13)* |
| **Kantorovich LP** | **quantum OT SDP** *(S13)* |
| **Kantorovich potentials $\varphi, \psi$** | **quantum potentials (SDP duals)** *(S13)* |
| **Gibbs kernel $K = e^{-C/\varepsilon}$** | **matrix-exponential kernel $e^{-C/\varepsilon}$** *(S14)* |
| **Sinkhorn iterations (matrix scaling)** | **Quantum Sinkhorn (matrix exp + partial trace fixed point)** *(S14)* |
| **Sinkhorn = KL projection (Amari)** | **Quantum Sinkhorn = Umegaki-relative-entropy projection** *(S14)* |

## 7. Your turn

1. **The ε bias.** For the small problem of §1, plot the Sinkhorn cost as a function of
   $\varepsilon \in [0.05, 5.0]$ and compare to the LP cost. At what $\varepsilon$ does
   the relative bias drop below $1\%$? *Hint: the bias scales like $\varepsilon \log(nm)$
   asymptotically.*
2. **Log-domain Sinkhorn.** At $\varepsilon = 0.01$ on the smooth-bumps example of §4,
   the standard (multiplicative) implementation will underflow because $K = e^{-C/\varepsilon}$
   has entries below $10^{-300}$. Rewrite the iteration in **log-space** — replacing
   matrix–vector products by `logsumexp` — and confirm you can reach $\varepsilon = 0.01$
   without `nan`s.
3. **Debiased Sinkhorn divergence.** Compute $S_\varepsilon(\mu, \nu) = \mathrm{OT}_\varepsilon(\mu, \nu)
   - \tfrac{1}{2}\mathrm{OT}_\varepsilon(\mu, \mu) - \tfrac{1}{2}\mathrm{OT}_\varepsilon(\nu, \nu)$
   (Feydy et al., 2019) on the bumps example and verify $S_\varepsilon(\mu, \mu) = 0$ and
   $S_\varepsilon(\mu, \nu) > 0$. Does it satisfy the triangle inequality on a few
   examples? *(It is a divergence, not a metric, but is much closer to a metric than the
   raw $\mathrm{OT}_\varepsilon$.)*

## Conclusions & references

- **LP duality** gives the OT problem two equivalent faces; for $c = |x - y|$ the dual
  is the **1-Lipschitz formula** $W_1 = \sup_{\mathrm{Lip}(f) \le 1} \int f\,\mathrm{d}(\mu - \nu)$.
- **Entropic regularization** makes the LP strongly convex and yields the **Sinkhorn
  matrix-scaling algorithm** — five lines, $\mathcal{O}(nm)$ per iteration, provably
  geometrically convergent.
- The **ε trade-off**: small $\varepsilon$ $\to$ sharp diagonal (LP), large $\varepsilon$
  $\to$ blurry independent coupling.
- **Amari's bridge.** The Sinkhorn plan is the **KL projection** of the Gibbs kernel
  $K = e^{-C/\varepsilon}$ onto the transportation polytope $T(a, b)$. M2 (KL geometry)
  and M3 (Wasserstein geometry) are the same geometry.
- **Next (S11 — Gaussians & dynamics):** the closed-form $W_2$ for Gaussians — the
  Bures–Wasserstein distance — which is the **direct bridge to density matrices** and
  opens M4.

**References:** M. Cuturi, "Sinkhorn distances", NeurIPS (2013); G. Peyré & M. Cuturi,
*Computational Optimal Transport* (NoW, 2019), chs. 4–5; S.-i. Amari, *Information
Geometry and Its Applications* (Springer, 2016), sec. 7.5; J. Feydy, T. Séjourné, F.-X.
Vialard, S. Amari, A. Trouvé, G. Peyré, "Interpolating between optimal transport and
MMD using Sinkhorn divergences", AISTATS (2019); R. Sinkhorn, "A relationship between
arbitrary positive matrices and doubly stochastic matrices", Ann. Math. Statist. 35,
876 (1964). Previous: `notebooks/03_ClassicalOptimalTransport/02_wasserstein.ipynb`.